# 01 - Data Validation

This notebook validates the price data and computes returns for the SGI project.

## Steps:
1. Load price data
2. Inspect for missing values
3. Compute returns
4. Run diagnostics
5. Save processed data

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_experiment_config
from src.data_loader import load_or_download_prices
from src.preprocessing import preprocess_data, validate_prices, validate_returns
from src.diagnostics import check_price_data_quality, check_return_data_quality
from src.paths import ensure_all_directories

ensure_all_directories()

## Load Configuration

In [ ]:
config = load_experiment_config('../configs/universe_sector_etfs.yaml')
print(f"Universe: {config['universe']['name']}")
print(f"Tickers: {config['universe']['tickers']}")
print(f"Date range: {config['data']['start_date']} to {config['data']['end_date']}")

## Load Price Data

In [ ]:
tickers = config['universe']['tickers']
start_date = config['data']['start_date']
end_date = config['data']['end_date']

prices = load_or_download_prices(tickers, start_date, end_date)
print(f"Price data shape: {prices.shape}")
prices.head()

## Check Data Quality

In [ ]:
quality = check_price_data_quality(prices)
print(f"Valid: {quality['is_valid']}")
print(f"Missing values: {quality['missing']['total']}")
print(f"Issues: {quality['issues']}")

In [ ]:
# Visualize prices
fig, ax = plt.subplots(figsize=(14, 6))
normalized = prices / prices.iloc[0] * 100
normalized.plot(ax=ax, alpha=0.7)
ax.set_title('Normalized Price Series (Base = 100)')
ax.set_ylabel('Price')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## Preprocess Data

In [ ]:
prices_clean, returns, metadata = preprocess_data(
    prices, 
    return_method='log',
    output_tag='sector_etfs'
)

print(f"Clean prices shape: {prices_clean.shape}")
print(f"Returns shape: {returns.shape}")

In [ ]:
# Return quality check
return_quality = check_return_data_quality(returns)
print(f"Return statistics:")
print(f"  Mean: {return_quality['statistics']['mean']:.6f}")
print(f"  Std: {return_quality['statistics']['std']:.6f}")
print(f"  Min: {return_quality['statistics']['min']:.4f}")
print(f"  Max: {return_quality['statistics']['max']:.4f}")

In [ ]:
# Return distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
returns.mean(axis=1).hist(bins=100, ax=axes[0], alpha=0.7)
axes[0].set_title('Distribution of Average Daily Returns')
axes[0].set_xlabel('Return')

# Rolling volatility
rolling_vol = returns.std(axis=1).rolling(20).mean() * np.sqrt(252)
rolling_vol.plot(ax=axes[1])
axes[1].set_title('Rolling 20-day Volatility (Annualized)')
axes[1].set_ylabel('Volatility')

plt.tight_layout()
plt.show()

## Summary

Data has been validated and processed. Outputs saved to `data/processed/`.